In [1]:
import os, pickle, tempfile
import pandas as pd

# Data

In [2]:
extra_set_path = "../../training_data/7.Extra_set"

In [3]:
with open(f"{extra_set_path}/features.pkl", "rb") as f:
    news = pickle.load(f)

len(news), news

(9,
 {'7gqu':     Residues                                                          \
           pdb label_entity_id label_asym_id label_seq_id auth_asym_id   
  0       7gqu               1             A           12            A   
  1       7gqu               1             A           13            A   
  2       7gqu               1             A           14            A   
  3       7gqu               1             A           15            A   
  4       7gqu               1             A           16            A   
  ..       ...             ...           ...          ...          ...   
  414     7gqu               1             A          426            A   
  415     7gqu               1             A          427            A   
  416     7gqu               1             A          428            A   
  417     7gqu               1             A          429            A   
  418     7gqu               1             A          430            A   
  
                       

In [4]:
with open(f"{extra_set_path}/news_sites.pkl", "rb") as f:
    news_sites = {k: v for k,v in pickle.load(f).items() if k in news}

len(news_sites), news_sites

(9,
 {'7gqu': [{'mod':      label_comp_id label_asym_id label_entity_id label_seq_id  \
    3420           X1L             D               4            .   
    
         pdbx_PDB_ins_code auth_seq_id auth_comp_id auth_asym_id  \
    3420                 ?        1002          X1L            A   
    
         pdbx_PDB_model_num pdbx_label_index pdbx_sifts_xref_db_name  \
    3420                  1             1002                       ?   
    
         pdbx_sifts_xref_db_acc pdbx_sifts_xref_db_num pdbx_sifts_xref_db_res  
    3420                      ?                      ?                      ?  ,
    'site':    label_comp_id label_asym_id label_entity_id label_seq_id pdbx_PDB_ins_code  \
    0            VAL             A               1           55                 ?   
    1            MET             A               1           56                 ?   
    2            ALA             A               1           57                 ?   
    3            THR             A     

In [5]:
assert all(len(sites) == 1 for sites in news_sites.values()), "Not all apos have a single annotated site"

In [6]:
models = pd.read_pickle("models.pkl")

models

{'model5': {'results': {'7gqu': {'pocket11': {'prob': 0.0006499344599433243,
     'pred': 0,
     'label': 0,
     'max_overlap': 0.16666666666666666,
     'pocket_in_site': 0.16666666666666666,
     'site_in_pocket': 0.06896551724137931},
    'pocket15': {'prob': 1.6904914446058683e-05,
     'pred': 0,
     'label': 0,
     'max_overlap': 0.0,
     'pocket_in_site': 0.0,
     'site_in_pocket': 0.0},
    'pocket7': {'prob': 1.0997085553299257e-07,
     'pred': 0,
     'label': 0,
     'max_overlap': 0.0,
     'pocket_in_site': 0.0,
     'site_in_pocket': 0.0},
    'pocket13': {'prob': 0.0004982067039236426,
     'pred': 0,
     'label': 0,
     'max_overlap': 0.0,
     'pocket_in_site': 0.0,
     'site_in_pocket': 0.0},
    'pocket14': {'prob': 2.0316012523835525e-05,
     'pred': 0,
     'label': 0,
     'max_overlap': 0.0,
     'pocket_in_site': 0.0,
     'site_in_pocket': 0.0},
    'pocket1': {'prob': 0.932378888130188,
     'pred': 1,
     'label': 1,
     'max_overlap': 0.89655172

In [7]:
colors = {
    "orange": "#D55E00".lower(),
    "green": "#009E73".lower(),
    "blue": "#0072B2".lower()
}

# Labelling

In [8]:
# Percentage of residues of "one" in "other"
get_overlap = lambda one, other: (
    len( one.merge(other) ) / len(one)
)

get_overlaps = lambda pdb, pocketd: {
    name: get_overlap(*one_in_other) 
        for site in news_sites[pdb] 
            for name, one_in_other in (
                ("pocket_in_site", (pocketd["residues"], site["site"])),
                ("site_in_pocket", (site["site"], pocketd["residues"])),
            )
}

def get_label(overlaps, site_in_pocket=None, pocket_in_site=None):
    assert not (site_in_pocket==None and pocket_in_site==None)
    
    if site_in_pocket is None:
        return int( overlaps["pocket_in_site"] >= pocket_in_site )
    if pocket_in_site is None:
        return int( overlaps["site_in_pocket"] >= site_in_pocket )
    return int( overlaps["site_in_pocket"] >= site_in_pocket or overlaps["pocket_in_site"] >= pocket_in_site )

In [9]:
def label_results(resultsd, site_in_pocket=0.65, pocket_in_site=None, prob_key=None):
    return pd.DataFrame((
        {
            "pdb": pdb,
            "pocket": pocket,
            **{"prob": pocketd[prob_key] for prob_key in (prob_key,) if prob_key is not None},
            "pred": pocketd["pred"],
            "label": get_label(overlaps, site_in_pocket, pocket_in_site),
            "max_overlap": max(overlaps.values()),
            **overlaps,
        }
        for pdb, pockets in resultsd.items()
        for pocket, pocketd in pockets.items()
        for overlaps in (get_overlaps(pdb, pocketd),)
    )).sort_values("max_overlap", ascending=False)

In [10]:
def label_our_results(resultsd):#, site_in_pocket=0.65, pocket_in_site=None):
    return pd.DataFrame((
        {
            "pdb": pdb,
            "pocket": pocket,
            "prob": pocketd["prob"],
            "pred": pocketd["pred"],
            "pred_top1": int( pocketd["prob"] == pdb_maxprob ),
            "label": pocketd["label"],
            "max_overlap": max(overlaps.values()),
            **overlaps,
        }
        for pdb, pockets in resultsd.items()
        for pdb_maxprob in (max(pktd["prob"] for pktd in pockets.values()),)
        for pocket, pocketd in pockets.items()
        for overlaps in ({k: pocketd[k] for k in ["pocket_in_site", "site_in_pocket"]},)
    )).sort_values("max_overlap", ascending=False)

In [11]:
for model, modeld in models.items():
    if model != "model5":
        models[model]["labelled"] = label_results(
            modeld["results"], 
            **modeld["labelling"], 
            prob_key=modeld["prob_key"]
        )
    else:
        models[model]["labelled"] = label_our_results(
            modeld["results"], 
            # **modeld["labelling"]
        )

# Pocket functions

In [12]:
import sys

sys.path.append("../../training_data")

In [13]:
from utils.utils import Cif, CifFileWriter
from utils.pocket_utils import Pocket

In [14]:
from biotite.structure.io.pdb import PDBFile

def get_pdb_atoms(f):
    atom_array = PDBFile.read(f).get_structure()
    return pd.DataFrame({
            "auth_asym_id": atom_array.chain_id,
            "auth_seq_id": atom_array.res_id,
            "auth_comp_id": atom_array.res_name,
            "auth_atom_id": atom_array.atom_name,
            "type_symbol": atom_array.element,
            "Cartn_x": atom_array.coord[0][:, 0],
            "Cartn_y": atom_array.coord[0][:, 1],
            "Cartn_z": atom_array.coord[0][:, 2],
            "pdbx_PDB_ins_code": (ic or '?' for ic in atom_array.ins_code)
        }, dtype=str)

In [15]:
def get_pocket(pocket_atoms, res_id, color):
    pocket_atoms["label_entity_id"] = '99'
    return {
        "pocketn": res_id,
        "atoms": pocket_atoms, 
        "representation": {
            "selection": [{'label_entity_id': '99', "auth_asym_id": pocket_atoms.auth_asym_id.unique().item(), 'auth_seq_id': int(res_id)}], 
            'color': colors[color]
        }
    }

In [16]:
def get_our_pocket(pdb, pocket, color):
    pocketn = pocket.replace('pocket', '')
    # pocket_atoms = (
    #     Cif(pdb, f"{extra_set_path}/pockets/{pdb}/{pdb}_out/{pdb}_out.cif", name=f"{pdb}_out")
    #     .atoms
    #     .query(f"label_comp_id == 'STP' and label_seq_id == '{pocketn}'")
    # )

    return {
        "pocketsf": f"{extra_set_path}/pockets/{pdb}/{pdb}_out/{pdb}_out.cif",
        "pocketn": pocketn,
        "pocket_sel": [{"label_comp_id": 'STP', "label_seq_id": int(pocketn)},],
        "color": colors[color]
    }#get_pocket(pocket_atoms, pocketn, color)
# {
#         "atoms": pocket_atoms, 
#         "representation": {
#             "selection": {'label_entity_id': '99', "auth_asym_id": pocket_atoms.auth_asym_id.unique().item(), 'auth_seq_id': int(pocketn)}, 
#             'color': colors[color]
#         }
#     }

models["model5"]["pocketf"] = get_our_pocket

In [17]:
def get_allositepro_pocket(pdb, pocket, color):
    resultsf = next(f for f in os.listdir(f"AllositePro/{pdb}") if f.endswith("_download"))
    # pockets are 0-indexed but residue numbers start at 1
    # also pocket0 can be residue ID 2, so all pockets auth_seq_id will be sorted and then the relevant residue id taken with the 0-index pocket number
    pockets_atoms = (
        get_pdb_atoms(f"AllositePro/{pdb}/{resultsf}/{resultsf.replace('_download', '')}.pdb")
        .query(f"auth_comp_id == 'STP'")
    )
    pocketn = sorted(pockets_atoms.auth_seq_id.unique())[ int(pocket.replace('pocket', '')) ]
    # pocket_atoms = pockets_atoms.query(f"auth_seq_id == '{pocketn}'")
    
    return {
        "pocketsf": f"AllositePro/{pdb}/{resultsf}/{resultsf.replace('_download', '')}.pdb",
        "pocketn": pocketn,
        "pocket_sel": [{"auth_comp_id": 'STP', "auth_seq_id": int(pocketn)},],
        "color": colors[color]
    }#get_pocket(pocket_atoms, pocketn, color)

models["allositepro"]["pocketf"] = get_allositepro_pocket

In [18]:
from functools import partial

In [19]:
def get_passer_pocket(pdb, pocket, color, model):
    pocketn = pocket.replace('pocket', '')
    # pocket_atoms = (
    #     get_pdb_atoms(f"PASSer/{model}/{pdb}/{pdb}_out.pdb")
    #     .query(f"auth_comp_id == 'STP' and auth_seq_id == '{pocketn}'")
    # )
    
    return {
        "pocketsf": f"PASSer/{model}/{pdb}/{pdb}_out.pdb",
        "pocketn": pocketn,
        "pocket_sel": [{"auth_comp_id": 'STP', "auth_seq_id": int(pocketn)},],
        "color": colors[color]
    }#get_pocket(pocket_atoms, pocketn, color)

models["passer_ensemble"]["pocketf"] = partial(get_passer_pocket, model="ensemble")
models["passer_automl"]["pocketf"] = partial(get_passer_pocket, model="automl")
models["passer_rank"]["pocketf"] = partial(get_passer_pocket, model="rank")

In [20]:
def get_allo_pocket(pdb, pocket, color):
    resultsf = next(f for f in os.listdir(f"ALLO/{pdb}") if f.startswith(f"{pdb}pdb") and f.endswith("_desc.txt")).replace("_desc.txt", "")
    pocket_atoms = (
        get_pdb_atoms(f"ALLO/{pdb}/pockets/{resultsf}_{pocket}_res.pdb")
    )
    
    # pocket_atoms["auth_asym_id"] = 'ZZZ'
    # pocket_atoms["label_entity_id"] = '99'
    return {
        "pocketsf": f"ALLO/{pdb}/pockets/{resultsf}_{pocket}_res.pdb",
        "pocketn": pocket,
        "pocket_sel": [
            {"auth_asym_id": atom['auth_asym_id'], "auth_seq_id": int(atom['auth_seq_id']), "auth_atom_id": atom['auth_atom_id']}
            for i, atom in pocket_atoms.query("auth_atom_id not in ['CA', 'C', 'O', 'N', 'CB']").iterrows()
        ],
        "color": colors[color]
    }
    #{
    #     "pocketn": pocket,
    #     "atoms": pocket_atoms.query("auth_atom_id not in ['CA', 'C', 'O', 'N', 'CB']"), 
    #     "representation": {
    #         "selection": [
    #             {"auth_asym_id": "ZZZ", 'auth_seq_id': int(res)}
    #             for res in pocket_atoms["auth_seq_id"].unique()
    #         ],
    #         'color': colors[color]
    #     }
    # }

models["allo"]["pocketf"] = get_allo_pocket

# View functions

In [21]:
import molviewspec as mvs
import json
from pathlib import Path

In [145]:
# ass_fields_list = ["_pdbx_struct_assembly", "_pdbx_struct_assembly_gen", "_pdbx_struct_oper_list"]

def view_pockets(pdb, model, pockets:list):
    cif = Cif(pdb, f"{extra_set_path}/origcifs/{pdb}_updated.cif.gz")

    # minimal_elements = lambda element="label_asym_id": site["site"][element].unique().tolist() + site["mod"][element].unique().tolist()

    site = news_sites[pdb][0]
    # atoms = cif.atoms.query(f"label_asym_id in {minimal_elements('label_asym_id')}")

    # # Fake entity data
    # entities = pd.concat((
    #     pd.DataFrame(cif.cif.data["_entity"], dtype=str),#.query(f"id in {minimal_elements('label_entity_id')}"),
    #     pd.DataFrame([{"id": "99", "type": "branched", "pdbx_description": "pockets"}]) # Fake the pockets as carbohydrates to manage their representation
    # )).fillna(".")

    

    # columns = list( set.intersection( *map(set, (pocket_atoms["atoms"].columns for pocket_atoms in pockets)) ) )
    # atoms = pd.concat((
    #     cif.atoms[columns],
    #     *(pocket_atoms["atoms"][columns] for pocket_atoms in pockets)
    # ))

    # with tempfile.NamedTemporaryFile("w+", suffix=".cif") as f:
    #     writer = CifFileWriter(f.name)
    #     writer.write({cif.entry_id.upper(): {
    #         "_entity": entities.to_dict(orient="list"),
    #         "_atom_site": atoms.to_dict(orient="list"),
    #         # **{k: cif.cif.data[k] for k in ass_fields_list}
    #     }})
    #     combined = Cif(pdb, filename=f.name)
    #     combined.cif.data # to cache it while 'f' exists

    
    builder = mvs.create_builder()
    structure = (
        builder.download(url=f"{pdb}.cif")
        .parse(format="mmcif")
        .model_structure()
    )
    assets = {f'{pdb}.cif': cif.cif.text.encode()}

    if custom_vizs[pdb]["camera"]["camera"]:
        builder.camera(
            **eval(
                "dict("
                + custom_vizs[pdb]["camera"]["camera"].strip()[1:-1].replace(":", "=") 
                + ")"
            )
        )
    if model in custom_vizs[pdb]["camera"]:
        builder.camera(
            **eval(
                "dict("
                + custom_vizs[pdb]["camera"][model].strip()[1:-1].replace(":", "=") 
                + ")"
            )
        )

    # Protein and site
    for auth_asym_id, siteres in site["site"].groupby("auth_asym_id"):
        protein = (
            structure
            .component(selector=[
                mvs.ComponentExpression(**sel)
                for sel in custom_vizs[pdb]["selections"].get(f"protein_{auth_asym_id}", [{"auth_asym_id": auth_asym_id},])
            ])
            .representation(type="cartoon", custom={"molstar_representation_params": {"ignoreLight": True}})#, custom={"ignoreLight": True})
            .color(color='#DADADA')
        )
        annos = f'annos_chain_{auth_asym_id}.json'
        assets[annos] = json.dumps(
            [
                {
                    'auth_asym_id': r["auth_asym_id"], 'auth_seq_id': int(r["auth_seq_id"]), 
                    'color': 'black' # 'pdbx_PDB_ins_code': r["pdbx_PDB_ins_code"],
                }
                for i, r in siteres.iterrows()
            ]
        ).encode()
        protein.color_from_uri(uri=annos, format='json', schema='all_atomic')
            
    # Ligands
    if not custom_vizs[pdb]["ligands"]["none"]:
        for entity_id in entities.query("type == 'non-polymer'").id.unique():
            if entity_id not in site["mod"].label_entity_id.unique():
                (
                    structure
                    .component(selector=mvs.ComponentExpression(label_entity_id=entity_id))
                    .representation(type="ball_and_stick", custom={"molstar_representation_params": {"ignoreLight": True}})
                    .color(color="white")
                )
    if model in custom_vizs[pdb]["ligands"]:
        for lig in custom_vizs[pdb]["ligands"][model]:
            (
                structure
                .component(selector=mvs.ComponentExpression(**lig))
                .representation(
                    type="ball_and_stick", 
                    size_factor=custom_vizs[pdb]["ligands"].get("size_factor", 0.6),
                    custom={"molstar_representation_params": {"ignoreLight": True}})
                .color(
                    custom={
                        "molstar_color_theme_name": "element-symbol",
                        "molstar_color_theme_params": {
                            "carbonColor": { "name": "uniform", "params": {"value": int("#808080".replace('#', ''), 16)} }
                        }
                    }
                )
                # .opacity(opacity=0.9)
            )
    
    # Modulator
    # for entity_id in entities.query("type == 'non-polymer'").id.unique():
    for entity_id in site["mod"].label_entity_id.unique():
        (
            structure
            .component(selector=mvs.ComponentExpression(label_entity_id=entity_id))
            .representation(
                type="ball_and_stick", 
                size_factor=custom_vizs[pdb]["ligands"].get("size_factor", 0.6),
                custom={"molstar_representation_params": {"ignoreLight": True}})
            .color(
                custom={
                    "molstar_color_theme_name": "element-symbol",
                    "molstar_color_theme_params": {
                        "carbonColor": { "name": "uniform", "params": {"value": int("#666666".replace('#', ''), 16)} }
                    }
                }
            )
            # .opacity(opacity=0.9)
        )



    # Pockets
    pockets_strs = {}
    for i, pocket in enumerate(pockets):
        pocketf = Path(pocket["pocketsf"])
        name = pocketf.name
        if name not in assets:
            pockets_strs[name] = (
                builder.download(url=name)
                .parse(format="mmcif" if pocketf.suffix == ".cif" else "pdb")
                .model_structure()
            )
            with open(pocketf, "r") as f:
                assets[name] = f.read().encode()
        
        (
            pockets_strs[name]
            .component(selector=[
                mvs.ComponentExpression(**sel)
                for sel in pocket["pocket_sel"]
            ])
            .representation(type="surface", custom={"molstar_representation_params": {"ignoreLight": True}})
            .color(color=pocket["color"])
            .opacity(
                opacity=(
                    custom_vizs[pdb]["pocket_opacity"]
                    .get(model, {pocket["pocketn"]: 0.5})
                    .get(pocket["pocketn"])
                )
            )
        )
    


    # View
    return mvs.MVSX(
        data=builder.get_state(),
        assets=assets
    )
    # v = mvsx.molstar_notebook(width= 1725, height= 875)
    # return v

In [146]:
def view_top(pdb, model, top=None):
    pocketf = models[model]["pocketf"]
    prob_key = models[model]["prob_key"]
    results = models[model]["results"][pdb]
    labelled = models[model]["labelled"].query(f"pdb == '{pdb}'").sort_values("prob", ascending=False)
    pos_pockets = labelled[labelled["label"] == 1]

    if top is None:
        top = labelled["label"].sum() or 1
    top_pockets = tuple(pocket for i, pocket in tuple(labelled.iterrows())[:top])

    for pocket in top_pockets:
        print(pocket["pocket"], {k: v for k, v in results[pocket["pocket"]].items() if k != "residues"}, "label:", pocket["label"])
        
    return view_pockets(
        pdb,
        model,
        tuple(
            [
                pocketf(pdb, pocket["pocket"], "green" if pocket["label"] == 1 else "blue")
                for pocket in top_pockets
            ] + [
                pocketf(pdb, pocket["pocket"], "orange")
                for i, pocket in pos_pockets.iterrows()
                if pocket["pocket"] not in (p["pocket"] for p in top_pockets)
            ]
        )
    )

# Viz

In [147]:
custom_vizs = {
    p: {
        "camera": {"camera": False}, # set a common camera after revision; also model-specific
        "ligands": {"none": False}, # set to True after revision; also model-specific; also size_factor
        "selections": {},
        "pocket_opacity": {}
    }
    for p in news.keys()
}


custom_vizs["7yg5"].update({
        "camera": {
            "camera": """{
    position: [95.49, 187.55, 251.44],
    target: [152.01, 159.35, 167.46],
    up: [0.84, 0.14, 0.52],
}""",
            "allositepro": """{
    position: [223.05, 143.6, 75.27],
    target: [152.01, 159.35, 167.46],
    up: [0.8, 0.15, 0.59],
}""",
            "passer_ensemble": """{
    position: [223.05, 143.6, 75.27],
    target: [152.01, 159.35, 167.46],
    up: [0.8, 0.15, 0.59],
}""",
        },
        "ligands": {
            "none": True,
            "size_factor": 0.7,
            "model5": [{"auth_asym_id": "A", "auth_seq_id": 2404},]
        },
        "pocket_opacity": {
            "allositepro": {"2": 0.7}
        }
    })


custom_vizs["7gqu"].update({
        "camera": {
            "camera": """{
    position: [77.37, 31.66, 32.78],
    target: [5.74, 27.52, 18.12],
    up: [0.04, 0.89, -0.46],
}""",
        },
        "ligands": {"none": True},
        "pocket_opacity": {
            "allositepro": {"2": 0.7}
        }
    })



custom_vizs["8aq6"].update({
        "camera": {
            "camera": """{
    position: [42.49, -12.34, 85.5],
    target: [44.34, -49.6, 65.39],
    up: [0.88, -0.19, 0.43],
}""",
        },
        "ligands": {"none": True},
        "pocket_opacity": {
            "allositepro": {"1": 0.8},
            "passer_ensemble": {"3": 0.8},
            "allo": {"P_2": 0.8}
        }
    })



custom_vizs["8f4s"].update({
        "camera": {
            "camera": """{
    position: [130.24, -13.79, 42.98],
    target: [94.2, 25.11, 24.15],
    up: [-0.39, 0.08, 0.92],
}""", # Establish the camera
        },
        "ligands": {"none": True},
        "selections": {
            "protein_A": [{"auth_asym_id": "A", "beg_auth_seq_id": 6799, "end_auth_seq_id": 7091},]
        }
    })


custom_vizs["8jp0"].update({
        "camera": {
            "camera": """{
    position: [103.33, 108.95, 69.94],
    target: [130.69, 134.14, 126.6],
    up: [-0.91, 0.09, 0.4],
}""", # Establish the camera
        },
        "ligands": {"none": True},
        "pocket_opacity": {
            "allositepro": {"2": 0.6}
        }
    })



custom_vizs["8qni"].update({
        "camera": {
            "camera": """{
    position: [31.03, 30.61, 94.01],
    target: [18.05, 10.31, 31.44],
    up: [0.18, 0.93, -0.34],
}""", # Establish the camera
        },
        "ligands": {"none": True},
    })


custom_vizs["8uk6"].update({
        "camera": {
            "camera": """{
    position: [111.85, 20.63, 87.19],
    target: [47.48, 25.38, 18.04],
    up: [0.43, -0.78, -0.46],
}""", # Establish the camera
        },
        "ligands": {"none": True},
    })




custom_vizs["8v81"].update({
        "camera": {
            "camera": """{
    position: [110.01, 224.39, 228.89],
    target: [150.45, 123.5, 140.12],
    up: [-0.62, 0.36, -0.69],
}""", # Establish the camera
        },
        "ligands": {
            "none": True, 
            "size_factor": 0.7,
            "passer_ensemble": [{"auth_asym_id": "A", "beg_auth_seq_id": 1501, "end_auth_seq_id": 1504},]
        },
        "pocket_opacity": {
            "allositepro": {"1": 0.7},
            # "passer_ensemble": {"49": 0.6}
        }
    })



custom_vizs["9dnm"].update({
        "camera": {
            "camera": """{
    position: [224.61, 233.28, 104.7],
    target: [205.02, 209.34, 179.72],
    up: [-0.7, 0.71, 0.05],
}""", # Establish the camera
        },
        "ligands": {
            "none": True, 
            "size_factor": 0.7
        },
        "selections": {
            "protein_A": [
                {"auth_asym_id": "A", "beg_auth_seq_id": 1, "end_auth_seq_id": 176},
                {"auth_asym_id": "A", "beg_auth_seq_id": 283, "end_auth_seq_id": 494},
            ]
        },
        "pocket_opacity": {
            "passer_ensemble": {"20": 0.7},
            "allo": {"P_0": 0.5}
        }
    })





- Components settings: Ignore light
- Examine ligands and make selection+component or hide all
- Duplicate spacefill (modulator), add stick-and-ball and hide spacefill. Carbon color uniform: ~30,30,30
- Adjust molecular surface Probe radius and Opacity (0.4)

In [148]:
news.keys()

dict_keys(['7gqu', '7yg5', '8aq6', '8f4s', '8jp0', '8qni', '8uk6', '8v81', '9dnm'])

In [149]:
modell = ["model5", "allositepro", "passer_ensemble", "allo"]#"allositepro",

In [150]:
i = 0

In [152]:
curr_model = modell[i]; print(curr_model)
i += 1
mvsx = view_top('8v81', curr_model); mvsx.molstar_notebook(width= 1100, height= 500)

allo
P_0 {'pred': 1, 'prob': 0.04905} label: 1
({'pocketsf': 'ALLO/8v81/pockets/8v81pdb5e0fdee8-99df-4852-a030-d2db77eee85a_P_0_res.pdb', 'pocketn': 'P_0', 'pocket_sel': [{'auth_asym_id': 'A', 'auth_seq_id': 92, 'auth_atom_id': 'CG'}, {'auth_asym_id': 'A', 'auth_seq_id': 92, 'auth_atom_id': 'CD'}, {'auth_asym_id': 'A', 'auth_seq_id': 92, 'auth_atom_id': 'OE1'}, {'auth_asym_id': 'A', 'auth_seq_id': 92, 'auth_atom_id': 'OE2'}, {'auth_asym_id': 'A', 'auth_seq_id': 95, 'auth_atom_id': 'CG'}, {'auth_asym_id': 'A', 'auth_seq_id': 95, 'auth_atom_id': 'CD'}, {'auth_asym_id': 'A', 'auth_seq_id': 95, 'auth_atom_id': 'CE'}, {'auth_asym_id': 'A', 'auth_seq_id': 95, 'auth_atom_id': 'NZ'}, {'auth_asym_id': 'A', 'auth_seq_id': 98, 'auth_atom_id': 'CG'}, {'auth_asym_id': 'A', 'auth_seq_id': 98, 'auth_atom_id': 'NE2'}, {'auth_asym_id': 'A', 'auth_seq_id': 99, 'auth_atom_id': 'CG'}, {'auth_asym_id': 'A', 'auth_seq_id': 99, 'auth_atom_id': 'CD'}, {'auth_asym_id': 'A', 'auth_seq_id': 131, 'auth_atom_id': 

<IPython.core.display.Javascript object>

In [1]:
mvsx.data.dict()

## Ignorelight tests